# Distribuciones y Muestreo

> Este notebook presenta los conceptos fundamentales que sustentan toda la inferencia estadística: la diferencia entre estadísticos y parámetros, las distribuciones de probabilidad más relevantes, y cómo se comporta una estimación cuando repetimos un experimento muchas veces.

---

## Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# Estilo de gráficos
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(42)

---

## 1. Estadístico vs Parámetro

Una de las distinciones más importantes en estadística es la diferencia entre lo que **calculamos a partir de datos** y lo que **realmente existe en la población**.

| Concepto | Definición | Ejemplo | Notación |
|---|---|---|---|
| **Parámetro** | Valor verdadero de la población (desconocido) | Media real de todos los clientes | $\mu$, $\sigma$, $p$ |
| **Estadístico** | Valor calculado a partir de una muestra | Media de 100 clientes encuestados | $\bar{x}$, $s$, $\hat{p}$ |

> 💡 **Idea clave:** nunca observamos los parámetros directamente. Los *estimamos* usando estadísticos calculados sobre muestras. Toda la inferencia estadística consiste en cuantificar cuánto confiamos en esa estimación.

<!-- IMAGEN SUGERIDA: diagrama que muestra una población grande con μ desconocido, una flecha de muestreo hacia una muestra pequeña, y x̄ calculado desde la muestra. Algo tipo: population → sample → statistic. -->

In [ ]:
# Simulamos una "población" de 100.000 personas con ingreso promedio real μ = 800.000
poblacion = np.random.normal(loc=800_000, scale=150_000, size=100_000)

# Parámetro real (en la práctica, esto es desconocido)
mu = poblacion.mean()
sigma = poblacion.std()
print(f"Parámetro real (μ): {mu:,.0f}")
print(f"Parámetro real (σ): {sigma:,.0f}")

# Tomamos una muestra de 100 personas
muestra = np.random.choice(poblacion, size=100, replace=False)

# Estadísticos de la muestra (esto es lo que tendríamos en la práctica)
x_bar = muestra.mean()
s = muestra.std(ddof=1)
print(f"\nEstadístico muestral (x̄): {x_bar:,.0f}")
print(f"Estadístico muestral (s): {s:,.0f}")
print(f"\nError de estimación: {abs(mu - x_bar):,.0f}")

---

## 2. Distribuciones Clave

Cuatro distribuciones de probabilidad aparecen repetidamente en inferencia estadística. Cada una tiene un rol específico según el tipo de problema.

<!-- IMAGEN SUGERIDA: panel 2x2 con las 4 distribuciones graficadas, mostrando sus formas características. Puede generarse con el código a continuación. -->

### 2.1 Distribución Normal

La distribución más fundamental en estadística. Aparece naturalmente cuando sumamos muchas variables aleatorias independientes (Teorema Central del Límite).

$$X \sim \mathcal{N}(\mu, \sigma^2)$$

**Características:**
- Simétrica respecto a la media $\mu$
- La dispersión está controlada por $\sigma$
- Regla 68-95-99.7: el 68% de los datos cae dentro de $\pm 1\sigma$, el 95% dentro de $\pm 2\sigma$

**Cuándo aparece:** distribución muestral de la media (n grande), errores de modelos de regresión.

In [ ]:
x = np.linspace(-4, 4, 300)

fig, ax = plt.subplots()
for mu_val, sigma_val, label in [(0, 1, 'N(0,1)'), (0, 0.5, 'N(0, 0.5)'), (1, 1, 'N(1, 1)')]:
    ax.plot(x, stats.norm.pdf(x, mu_val, sigma_val), label=label, lw=2)

ax.set_title('Distribución Normal')
ax.set_xlabel('x')
ax.set_ylabel('Densidad')
ax.legend()
plt.tight_layout()
plt.show()

### 2.2 Distribución t-Student

Una distribución similar a la Normal pero con **colas más pesadas**. Se usa cuando estimamos la media de una población y **no conocemos $\sigma$** (la desviación estándar poblacional), lo que ocurre en casi todos los casos reales.

$$T = \frac{\bar{X} - \mu}{s / \sqrt{n}} \sim t_{n-1}$$

**Características:**
- Parametrizada por los **grados de libertad** ($\nu = n-1$)
- A medida que $\nu \to \infty$, se aproxima a la Normal estándar
- Con muestras pequeñas ($n < 30$), las colas más pesadas reflejan mayor incertidumbre

**Cuándo aparece:** pruebas t, intervalos de confianza para la media, inferencia sobre coeficientes de regresión.

In [ ]:
x = np.linspace(-5, 5, 300)

fig, ax = plt.subplots()
ax.plot(x, stats.norm.pdf(x), label='Normal(0,1)', lw=2, linestyle='--', color='black')
for df, color in [(1, 'red'), (5, 'orange'), (30, 'steelblue')]:
    ax.plot(x, stats.t.pdf(x, df), label=f't (gl={df})', lw=2, color=color)

ax.set_title('t-Student vs Normal: efecto de los grados de libertad')
ax.set_xlabel('x')
ax.set_ylabel('Densidad')
ax.legend()
plt.tight_layout()
plt.show()

### 2.3 Distribución Chi-cuadrado ($\chi^2$)

Surge cuando **elevamos al cuadrado** variables normales estándar y las sumamos. Es siempre positiva y asimétrica hacia la derecha.

$$\chi^2_k = Z_1^2 + Z_2^2 + \cdots + Z_k^2, \quad Z_i \sim \mathcal{N}(0,1)$$

**Características:**
- Solo toma valores positivos
- Parametrizada por grados de libertad $k$
- A mayor $k$, más simétrica (se aproxima a Normal)

**Cuándo aparece:** inferencia sobre varianzas, pruebas de bondad de ajuste, pruebas de independencia en tablas de contingencia.

In [ ]:
x = np.linspace(0, 30, 300)

fig, ax = plt.subplots()
for k, color in [(1, 'red'), (3, 'orange'), (5, 'green'), (10, 'steelblue')]:
    ax.plot(x, stats.chi2.pdf(x, k), label=f'χ² (k={k})', lw=2, color=color)

ax.set_title('Distribución Chi-cuadrado')
ax.set_xlabel('x')
ax.set_ylabel('Densidad')
ax.set_ylim(0, 0.5)
ax.legend()
plt.tight_layout()
plt.show()

### 2.4 Distribución F

Es el cociente de dos distribuciones $\chi^2$ divididas por sus grados de libertad. También es positiva y asimétrica.

$$F = \frac{\chi^2_{d_1}/d_1}{\chi^2_{d_2}/d_2}$$

**Características:**
- Parametrizada por dos grados de libertad ($d_1$, $d_2$)
- Solo toma valores positivos

**Cuándo aparece:** prueba F en regresión lineal múltiple (¿el modelo en su conjunto es significativo?), comparación de varianzas entre grupos (ANOVA).

In [ ]:
x = np.linspace(0, 5, 300)

fig, ax = plt.subplots()
for d1, d2, color in [(1, 1, 'red'), (2, 5, 'orange'), (5, 10, 'green'), (10, 30, 'steelblue')]:
    ax.plot(x, stats.f.pdf(x, d1, d2), label=f'F ({d1},{d2})', lw=2, color=color)

ax.set_title('Distribución F')
ax.set_xlabel('x')
ax.set_ylabel('Densidad')
ax.set_ylim(0, 1.6)
ax.legend()
plt.tight_layout()
plt.show()

### Resumen: ¿cuándo usar cada distribución?

| Distribución | Aparece en... |
|---|---|
| Normal | Distribución muestral (n grande), errores de regresión |
| t-Student | Pruebas sobre medias, coeficientes de regresión |
| Chi-cuadrado | Pruebas sobre varianzas, tablas de contingencia |
| F | Prueba global de regresión, ANOVA |

---

## 3. Distribución Muestral de la Media

Si tomamos **muchas muestras distintas** de la misma población y calculamos la media de cada una, ¿cómo se distribuyen esas medias?

Esta es la pregunta central de la **distribución muestral**: no es la distribución de los datos individuales, sino la distribución del *estadístico* calculado sobre muchas muestras.

<!-- IMAGEN SUGERIDA: diagrama con una población, múltiples flechas de muestreo hacia muestras distintas, cada una con su propia x̄, y todas esas x̄ formando una nueva distribución. -->

**Resultado teórico:** si la población tiene media $\mu$ y desviación estándar $\sigma$, entonces la media muestral $\bar{X}$ de muestras de tamaño $n$ satisface:

$$\bar{X} \sim \mathcal{N}\left(\mu,\ \frac{\sigma^2}{n}\right)$$

Es decir:
- La media de $\bar{X}$ es la misma que la de la población: $E[\bar{X}] = \mu$ → el estimador es **insesgado**
- La varianza de $\bar{X}$ es $\sigma^2/n$ → a mayor $n$, más precisa la estimación

In [ ]:
# Población asimétrica (no normal)
poblacion = np.random.exponential(scale=1.0, size=100_000)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Mostrar la distribución de la población
axes[0].hist(poblacion, bins=80, color='steelblue', edgecolor='white', density=True)
axes[0].set_title('Población original (Exponencial)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('Densidad')

# Distribución muestral para n=5 y n=30
for ax, n, color, title in [
    (axes[1], 5,  'salmon',    'Distribución muestral\n(n = 5)'),
    (axes[2], 30, 'seagreen',  'Distribución muestral\n(n = 30)')
]:
    medias = [np.random.choice(poblacion, size=n).mean() for _ in range(5000)]
    ax.hist(medias, bins=60, color=color, edgecolor='white', density=True)
    ax.axvline(np.mean(medias), color='black', linestyle='--', lw=1.5, label=f'Media = {np.mean(medias):.2f}')
    ax.set_title(title)
    ax.set_xlabel('x̄')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---

## 4. Teorema Central del Límite (simulación)

El **Teorema Central del Límite (TCL)** es uno de los resultados más poderosos de la estadística:

> *Bajo condiciones generales, la distribución muestral de la media converge a una Normal a medida que el tamaño de muestra $n$ aumenta, independientemente de la forma de la distribución original.*

Esto es lo que permite usar la distribución Normal (o t-Student) en pruebas de hipótesis e intervalos de confianza, incluso cuando los datos originales no son normales.

<!-- IMAGEN SUGERIDA: cuatro paneles mostrando cómo la distribución muestral se va acercando a una campana para n=1, 5, 30, 100 —el código a continuación lo genera. -->

In [ ]:
# Población muy asimétrica: distribución uniforme
poblacion = np.random.uniform(0, 1, size=100_000)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
tamaños = [1, 5, 30, 100]
colores = ['#e74c3c', '#e67e22', '#27ae60', '#2980b9']

for ax, n, color in zip(axes, tamaños, colores):
    medias = [np.random.choice(poblacion, size=n).mean() for _ in range(5000)]
    ax.hist(medias, bins=50, density=True, color=color, edgecolor='white', alpha=0.8)
    
    # Superponemos la curva normal teórica
    mu_teo = np.mean(poblacion)
    se_teo = np.std(poblacion) / np.sqrt(n)
    x_range = np.linspace(min(medias), max(medias), 200)
    ax.plot(x_range, stats.norm.pdf(x_range, mu_teo, se_teo), 'k--', lw=2)
    
    ax.set_title(f'n = {n}', fontsize=13)
    ax.set_xlabel('x̄')
    if n == 1:
        ax.set_ylabel('Densidad')

plt.suptitle('TCL: distribución muestral de la media (población Uniforme)', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

**Observaciones:**
- Con $n=1$, la distribución muestral hereda la forma de la población (uniforme)
- Con $n=5$, empieza a perfilarse una forma de campana
- Con $n=30$, la aproximación Normal es bastante buena
- Con $n=100$, la distribución muestral es casi perfectamente Normal

> ⚠️ **Regla práctica:** $n \geq 30$ generalmente es suficiente para que el TCL sea una buena aproximación, aunque esto depende de cuán asimétrica sea la distribución original.

---

## 5. Error Estándar

El **error estándar (SE)** mide la dispersión de la distribución muestral. En otras palabras, cuantifica cuánto varía el estadístico de muestra en muestra.

Para la media muestral:

$$SE(\bar{X}) = \frac{\sigma}{\sqrt{n}}$$

Si no conocemos $\sigma$ (lo que ocurre en la práctica), lo estimamos con $s$:

$$\widehat{SE}(\bar{X}) = \frac{s}{\sqrt{n}}$$

**Dos mensajes clave:**
1. A mayor $n$, menor error estándar → estimaciones más precisas
2. El error estándar cae como $1/\sqrt{n}$: para reducir el SE a la mitad, necesitas **cuadruplicar** el tamaño de muestra

<!-- IMAGEN SUGERIDA: gráfico de SE vs n mostrando la curva 1/√n, destacando el rendimiento decreciente de agregar observaciones. -->

In [ ]:
# Efecto del tamaño de muestra sobre el error estándar
sigma = 150_000  # desviación estándar poblacional
tamaños_n = np.arange(5, 500, 5)
errores_estandar = sigma / np.sqrt(tamaños_n)

fig, ax = plt.subplots()
ax.plot(tamaños_n, errores_estandar, color='steelblue', lw=2.5)
ax.axhline(sigma / np.sqrt(100), color='red', linestyle='--', lw=1.5, label=f'n=100: SE={sigma/np.sqrt(100):,.0f}')
ax.axhline(sigma / np.sqrt(400), color='green', linestyle='--', lw=1.5, label=f'n=400: SE={sigma/np.sqrt(400):,.0f}')
ax.set_title('Error estándar de la media según tamaño de muestra')
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('Error estándar')
ax.legend()
plt.tight_layout()
plt.show()

print("Para reducir el SE a la mitad (de n=100 a n=400), necesitamos 4x más datos.")

In [ ]:
# Verificación empírica: SE teórico vs SE observado en simulación
poblacion = np.random.normal(loc=800_000, scale=150_000, size=100_000)

resultados = []
for n in [10, 30, 100, 300]:
    medias_sim = [np.random.choice(poblacion, size=n).mean() for _ in range(3000)]
    se_observado = np.std(medias_sim)
    se_teorico = poblacion.std() / np.sqrt(n)
    resultados.append({'n': n, 'SE teórico': round(se_teorico, 1), 'SE observado': round(se_observado, 1)})

pd.DataFrame(resultados).set_index('n')

---

## Resumen

| Concepto | Idea central |
|---|---|
| Parámetro vs Estadístico | El parámetro es desconocido; el estadístico lo estima desde una muestra |
| Normal | Distribución base; describe el comportamiento de muchos fenómenos y estimadores |
| t-Student | Versión robusta de la Normal cuando no conocemos $\sigma$; colas más pesadas |
| Chi-cuadrado | Surge al cuadrar normales; aparece en varianzas y tablas de contingencia |
| F | Cociente de chi-cuadrados; aparece en la prueba global de regresión |
| Distribución muestral | Distribución del estadístico al repetir el muestreo; tiene media $\mu$ y dispersión $\sigma/\sqrt{n}$ |
| TCL | Con $n$ suficientemente grande, la media muestral es aproximadamente Normal |
| Error estándar | Mide la precisión del estimador; cae como $1/\sqrt{n}$ |

---

## Referencias

1. Wackerly, D., Mendenhall, W., Scheaffer, R. (2008). *Mathematical Statistics with Applications*. Thomson.
2. OpenStax (2022). [*Introductory Statistics*](https://openstax.org/books/introductory-statistics/pages/1-introduction).
3. SciPy Stats Documentation: [https://docs.scipy.org/doc/scipy/reference/stats.html](https://docs.scipy.org/doc/scipy/reference/stats.html)